# Deep Momentum — Full Pipeline Run

Replication of Han & Qin (2026), "Bimodality Everywhere: International Evidence of Deep Momentum"

Pilot run: US, Canada, Australia

In [ ]:
import warnings
warnings.filterwarnings('ignore')

PILOT_COUNTRIES = ['US', 'TO', 'AX']  # United States, Canada, Australia
N_ENSEMBLE = 100  # paper uses 100
OOS_START = '2010-01-31'  # paper's OOS start

print('Pilot countries:', PILOT_COUNTRIES)
print(f'Ensemble size: {N_ENSEMBLE}')

## Step 1: Data Fetch

In [ ]:
from data_fetch import fetch_all_countries

raw_data = fetch_all_countries(
    suffixes=PILOT_COUNTRIES,
    max_stocks_per_country=None,  # all stocks
    min_date='1990-01-01',
    save=True,
)

## Step 2: Data Filtering

In [ ]:
from data_filter import filter_all_countries

filtered_data = filter_all_countries(save=True)

## Step 3: Feature Construction

In [ ]:
from features import build_features
from config import COUNTRIES, CACHE_DIR
from pathlib import Path
import pandas as pd

cache_dir = Path(CACHE_DIR)
features_data = {}

for suffix in PILOT_COUNTRIES:
    filtered_path = cache_dir / f'filtered_{suffix}.parquet'
    if not filtered_path.exists():
        print(f'No filtered data for {suffix}, skipping')
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df = pd.read_parquet(filtered_path)
    df, feature_cols = build_features(df, country_name)
    
    # Save
    df.to_parquet(cache_dir / f'features_{suffix}.parquet', index=False)
    features_data[suffix] = (df, feature_cols)
    
print(f'\nFeatures built for {len(features_data)} countries')
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

## Step 4: Model Training + Prediction

In [ ]:
from model import run_walk_forward

predictions_data = {}

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    
    print(f'\n{"="*60}')
    print(f'TRAINING: {country_name} (.{suffix})')
    print(f'{"="*60}')
    
    predictions = run_walk_forward(
        df, feature_cols,
        n_ensemble=N_ENSEMBLE,
        verbose=True,
    )
    
    if not predictions.empty:
        predictions.to_parquet(cache_dir / f'predictions_{suffix}.parquet', index=False)
        predictions_data[suffix] = predictions
        print(f'  Saved {len(predictions)} predictions')
    else:
        print(f'  No predictions generated')

print(f'\nPredictions generated for {len(predictions_data)} countries')

## Step 5: Portfolio Construction + Performance

In [ ]:
from portfolio import run_all_strategies, print_performance_table

all_results = {}

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data or suffix not in predictions_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    predictions = predictions_data[suffix]
    
    # Merge fwd_return into predictions
    fwd = df[['symbol', 'date', 'fwd_return']].dropna()
    predictions = predictions.drop(columns=['fwd_return'], errors='ignore')
    predictions = predictions.merge(fwd, on=['symbol', 'date'], how='left')
    
    print(f'\n{"="*60}')
    print(f'{country_name}')
    print(f'{"="*60}')
    
    results = run_all_strategies(df, predictions, oos_start=OOS_START)
    print_performance_table(results)
    all_results[suffix] = results

## Step 6: Full Report + Metrics

In [ ]:
from metrics import full_report

for suffix in PILOT_COUNTRIES:
    if suffix not in features_data or suffix not in predictions_data:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    df, feature_cols = features_data[suffix]
    predictions = predictions_data[suffix]
    
    # Merge fwd_return
    fwd = df[['symbol', 'date', 'fwd_return']].dropna()
    preds = predictions.drop(columns=['fwd_return'], errors='ignore')
    preds = preds.merge(fwd, on=['symbol', 'date'], how='left')
    
    full_report(df, preds, all_results.get(suffix, {}), country_name)

## Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for suffix in PILOT_COUNTRIES:
    if suffix not in all_results:
        continue
    
    _, country_name, _, _ = COUNTRIES[suffix]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Panel 1: Cumulative returns
    ax = axes[0]
    for name, color in [('MOM', '#1f77b4'), ('XGB', '#ff7f0e'), ('RET', '#2ca02c'), ('SRP', '#d62728')]:
        if name in all_results[suffix] and not all_results[suffix][name]['portfolio'].empty:
            port = all_results[suffix][name]['portfolio']
            cum = (1 + port['ls_ret']).cumprod()
            ax.plot(port['date'], cum, label=name, color=color, linewidth=1.2)
    
    ax.set_ylabel('Cumulative Return (1 = start)')
    ax.set_title(f'{country_name} — Cumulative L/S Returns')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_yscale('log')
    
    # Panel 2: Rolling 12-month return
    ax = axes[1]
    for name, color in [('MOM', '#1f77b4'), ('XGB', '#ff7f0e'), ('RET', '#2ca02c')]:
        if name in all_results[suffix] and not all_results[suffix][name]['portfolio'].empty:
            port = all_results[suffix][name]['portfolio']
            rolling = port['ls_ret'].rolling(12).mean() * 12
            ax.plot(port['date'], rolling, label=f'{name} (rolling 12m ann.)', 
                    color=color, linewidth=1)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('Annualized Return')
    ax.set_title(f'{country_name} — Rolling 12-Month Return')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'results/{suffix}_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## Cross-Country Comparison (Table 5 equivalent)

In [ ]:
print(f'{"Country":<15s} {"":>5s} {"MOM":>12s} {"XGB":>12s} {"RET":>12s} {"SRP":>12s}')
print(f'{"":>20s} {"Ret / Sharpe":>12s} {"Ret / Sharpe":>12s} {"Ret / Sharpe":>12s} {"Ret / Sharpe":>12s}')
print('-' * 75)

for suffix in PILOT_COUNTRIES:
    if suffix not in all_results:
        continue
    _, country_name, _, _ = COUNTRIES[suffix]
    
    parts = [f'{country_name:<15s}', f'{"":>5s}']
    for name in ['MOM', 'XGB', 'RET', 'SRP']:
        if name in all_results[suffix] and all_results[suffix][name]['metrics']:
            m = all_results[suffix][name]['metrics']
            parts.append(f'{m["mean_annual"]:>5.1%}/{m["sharpe"]:>5.2f}')
        else:
            parts.append(f'{"N/A":>12s}')
    
    print('  '.join(parts))